In [ ]:
import zipfile
import os

zip_path = "/content/collections_30k_dataset.zip"
extract_path = "/content/credresolve_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Files extracted successfully!")
print(os.listdir(extract_path))

Files extracted successfully!
['call_dispositions.csv', 'sms_events.csv', 'payments.csv', 'accounts.csv', 'agent_sessions.csv', 'README.md', 'account_status_history.csv', 'field_visits.csv', 'call_attempts.csv', 'calls.csv', 'agents.csv', 'complaints.csv', 'borrowers.csv', 'campaigns.csv', 'vendor_telephony.csv', 'daily_targeting.csv', 'whatsapp_events.csv', 'data_dictionary.csv', 'promises_to_pay.csv']


In [ ]:
readme_path = "/content/credresolve_data/README.md"

with open(readme_path, "r", encoding="utf-8") as f:
    readme = f.read()

print(readme)

# Synthetic Collections Analytics Dataset

Generated for a data-analyst challenge. Seed = 42.

The package contains 17 relational datasets and intentionally includes:
- duplicates
- missing values
- conflicting timestamps
- UTC / Asia/Kolkata / Asia/Dubai timestamps
- inconsistent identifiers
- late-arriving events
- schema versions
- legacy disposition codes
- duplicate payment references/events
- overwritten-style status history
- multiple agent identifiers
- inconsistent campaign definitions

Important: event-table row counts are intentionally different. Some tables exceed their nominal size because duplicate rows were injected.



In [ ]:
import pandas as pd

dictionary = pd.read_csv(
    "/content/credresolve_data/data_dictionary.csv"
)

print("Shape:", dictionary.shape)

display(dictionary)

Shape: (143, 3)


,dataset,column,dtype
0,borrowers,borrower_id,object
1,borrowers,name,object
2,borrowers,phone,object
3,borrowers,email,object
4,borrowers,city,object
...,...,...,...
138,account_status_history,event_at,datetime64[ns]
139,account_status_history,status,object
140,account_status_history,changed_by,object
141,account_status_history,source,object


In [ ]:
import glob
import os
import pandas as pd

files = glob.glob("/content/credresolve_data/*.csv")

summary = []

for file in files:
    df = pd.read_csv(file)

    summary.append({
        "Dataset": os.path.basename(file),
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

dataset_summary = pd.DataFrame(summary)

display(dataset_summary.sort_values("Rows", ascending=False))

,Dataset,Rows,Columns
7,call_attempts.csv,120000,9
8,calls.csv,91350,11
15,whatsapp_events.csv,60600,8
5,account_status_history.csv,60000,8
14,daily_targeting.csv,45000,7
1,sms_events.csv,45000,8
0,call_dispositions.csv,35000,8
11,borrowers.csv,30600,8
9,agents.csv,30000,8
3,accounts.csv,30000,11


In [ ]:
display(
    dataset_summary.sort_values("Dataset").reset_index(drop=True)
)

,Dataset,Rows,Columns
0,account_status_history.csv,60000,8
1,accounts.csv,30000,11
2,agent_sessions.csv,15000,7
3,agents.csv,30000,8
4,borrowers.csv,30600,8
5,call_attempts.csv,120000,9
6,call_dispositions.csv,35000,8
7,calls.csv,91350,11
8,campaigns.csv,120,7
9,complaints.csv,8000,9


In [ ]:
for file in sorted(files):
    df = pd.read_csv(file)

    print("\n" + "=" * 80)
    print("DATASET:", os.path.basename(file))
    print("SHAPE:", df.shape)
    print("COLUMNS:")

    for col in df.columns:
        print(" -", col)


DATASET: account_status_history.csv
SHAPE: (60000, 8)
COLUMNS:
 - history_id
 - account_id
 - borrower_id
 - event_at
 - status
 - changed_by
 - source
 - recorded_at

DATASET: accounts.csv
SHAPE: (30000, 11)
COLUMNS:
 - account_id
 - borrower_id
 - loan_type
 - principal_amount
 - outstanding_amount
 - dpd
 - risk_segment
 - status
 - opened_at
 - timezone
 - schema_version

DATASET: agent_sessions.csv
SHAPE: (15000, 7)
COLUMNS:
 - session_id
 - agent_id
 - login_at
 - channel
 - device_id
 - timezone
 - logout_at

DATASET: agents.csv
SHAPE: (30000, 8)
COLUMNS:
 - agent_id
 - employee_code
 - agent_name
 - vendor_id
 - team
 - status
 - joined_at
 - updated_at

DATASET: borrowers.csv
SHAPE: (30600, 8)
COLUMNS:
 - borrower_id
 - name
 - phone
 - email
 - city
 - created_at
 - updated_at
 - state

DATASET: call_attempts.csv
SHAPE: (120000, 9)
COLUMNS:
 - attempt_id
 - account_id
 - borrower_id
 - event_at
 - call_id
 - agent_id
 - attempt_no
 - vendor_id
 - attempt_status

DATASET: cal

In [ ]:
import pandas as pd
import glob
import os

data = {}

files = glob.glob("/content/credresolve_data/*.csv")

for file in files:
    name = os.path.splitext(os.path.basename(file))[0]
    data[name] = pd.read_csv(file)

print("Total datasets loaded:", len(data))
print("\nDataset names:")

for name in sorted(data.keys()):
    print("-", name)

Total datasets loaded: 18

Dataset names:
- account_status_history
- accounts
- agent_sessions
- agents
- borrowers
- call_attempts
- call_dispositions
- calls
- campaigns
- complaints
- daily_targeting
- data_dictionary
- field_visits
- payments
- promises_to_pay
- sms_events
- vendor_telephony
- whatsapp_events


In [ ]:
id_columns = {
    "borrowers": "borrower_id",
    "accounts": "account_id",
    "agents": "agent_id",
    "payments": "payment_id",
    "calls": "call_id",
    "call_attempts": "attempt_id",
    "campaigns": "campaign_id",
    "promises_to_pay": "ptp_id",
    "field_visits": "visit_id",
    "complaints": "complaint_id",
    "agent_sessions": "session_id",
    "account_status_history": "history_id"
}

for table, column in id_columns.items():
    if table in data and column in data[table].columns:
        total = len(data[table])
        unique = data[table][column].nunique()
        duplicates = total - unique

        print(
            f"{table:25} | "
            f"Rows: {total:7} | "
            f"Unique IDs: {unique:7} | "
            f"Duplicate IDs: {duplicates:6}"
        )

borrowers                 | Rows:   30600 | Unique IDs:   11015 | Duplicate IDs:  19585
accounts                  | Rows:   30000 | Unique IDs:   30000 | Duplicate IDs:      0
agents                    | Rows:   30000 | Unique IDs:    1000 | Duplicate IDs:  29000
payments                  | Rows:   25500 | Unique IDs:   25000 | Duplicate IDs:    500
calls                     | Rows:   91350 | Unique IDs:   90000 | Duplicate IDs:   1350
call_attempts             | Rows:  120000 | Unique IDs:  120000 | Duplicate IDs:      0
campaigns                 | Rows:     120 | Unique IDs:     120 | Duplicate IDs:      0
promises_to_pay           | Rows:   18000 | Unique IDs:   18000 | Duplicate IDs:      0
field_visits              | Rows:   25000 | Unique IDs:   25000 | Duplicate IDs:      0
complaints                | Rows:    8000 | Unique IDs:    8000 | Duplicate IDs:      0
agent_sessions            | Rows:   15000 | Unique IDs:   15000 | Duplicate IDs:      0
account_status_history    | Rows

In [ ]:
missing_summary = []

for name, df in data.items():

    total_missing = df.isna().sum().sum()

    missing_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": total_missing
    })

missing_df = pd.DataFrame(missing_summary)

display(
    missing_df.sort_values(
        "Missing Values",
        ascending=False
    )
)

,Dataset,Rows,Columns,Missing Values
7,call_attempts,120000,9,2400
8,calls,91350,11,1827
11,borrowers,30600,8,1509
3,accounts,30000,11,455
2,payments,25500,9,382
6,field_visits,25000,10,250
1,sms_events,45000,8,0
0,call_dispositions,35000,8,0
5,account_status_history,60000,8,0
4,agent_sessions,15000,7,0


In [ ]:
for name, df in data.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\n" + "=" * 60)
        print("DATASET:", name)
        print(missing)


DATASET: payments
payment_reference    382
dtype: int64

DATASET: accounts
borrower_id    455
dtype: int64

DATASET: field_visits
scheduled_at    250
dtype: int64

DATASET: call_attempts
vendor_id    2400
dtype: int64

DATASET: calls
agent_id    1827
dtype: int64

DATASET: borrowers
phone    614
email    895
dtype: int64


In [ ]:
id_columns = {
    "borrowers": "borrower_id",
    "accounts": "account_id",
    "agents": "agent_id",
    "payments": "payment_id",
    "calls": "call_id",
    "call_attempts": "attempt_id",
    "campaigns": "campaign_id",
    "promises_to_pay": "ptp_id",
    "field_visits": "visit_id",
    "complaints": "complaint_id",
    "agent_sessions": "session_id",
    "account_status_history": "history_id"
}

duplicate_summary = []

for table, column in id_columns.items():

    if table in data and column in data[table].columns:

        df = data[table]

        total_rows = len(df)
        unique_ids = df[column].nunique(dropna=True)
        duplicate_rows = total_rows - unique_ids

        duplicate_summary.append({
            "Dataset": table,
            "ID Column": column,
            "Total Rows": total_rows,
            "Unique IDs": unique_ids,
            "Potential Duplicate Rows": duplicate_rows
        })

duplicate_df = pd.DataFrame(duplicate_summary)

display(duplicate_df)

,Dataset,ID Column,Total Rows,Unique IDs,Potential Duplicate Rows
0,borrowers,borrower_id,30600,11015,19585
1,accounts,account_id,30000,30000,0
2,agents,agent_id,30000,1000,29000
3,payments,payment_id,25500,25000,500
4,calls,call_id,91350,90000,1350
5,call_attempts,attempt_id,120000,120000,0
6,campaigns,campaign_id,120,120,0
7,promises_to_pay,ptp_id,18000,18000,0
8,field_visits,visit_id,25000,25000,0
9,complaints,complaint_id,8000,8000,0


In [ ]:
borrower_counts = (
    data["borrowers"]
    ["borrower_id"]
    .value_counts()
)

print("Total unique borrowers:", borrower_counts.nunique())
print("Number of borrower IDs appearing more than once:",
      (borrower_counts > 1).sum())

display(
    borrower_counts
    .loc[borrower_counts > 1]
    .head(20)
)

Total unique borrowers: 11
Number of borrower IDs appearing more than once: 8566


,count
borrower_id,
BRW0008879,11
BRW0006302,11
BRW0002520,10
BRW0001237,10
BRW0005385,10
BRW0001843,9
BRW0003622,9
BRW0005630,9
BRW0002013,9


In [ ]:
repeated_borrower = borrower_counts[borrower_counts > 1].index[0]

print("Example repeated borrower_id:", repeated_borrower)

display(
    data["borrowers"][
        data["borrowers"]["borrower_id"] == repeated_borrower
    ]
)

Example repeated borrower_id: BRW0008879


,borrower_id,name,phone,email,city,created_at,updated_at,state
1280,BRW0008879,Priya Mehta,9.845814e+09,user18232@example.com,Bengaluru,2026-04-21 18:56:59,2026-04-18 12:09:43,Karnataka
3194,BRW0008879,Pooja Nair,9.461372e+09,user19616@example.com,Hyderabad,2025-12-18 09:31:29,2026-01-19 19:52:27,Telangana
10580,BRW0008879,Neha Singh,9.506041e+09,user12411@example.com,Delhi,2025-05-25 08:15:49,2026-01-30 10:41:38,Delhi
10600,BRW0008879,Neha Singh,9.146559e+09,NaN,Kolkata,2026-06-10 00:05:51,2025-07-30 00:22:40,West Bengal
13433,BRW0008879,Rahul Verma,9.203332e+09,user12489@example.com,Chennai,2025-12-26 16:13:31,2026-03-26 15:42:12,Tamil Nadu
13827,BRW0008879,Vikram Shah,9.855081e+09,user17960@example.com,Chennai,2026-03-12 19:24:27,2025-12-07 21:36:14,Tamil Nadu
15176,BRW0008879,Aarav Sharma,9.585574e+09,user16090@example.com,Jaipur,2026-04-03 22:34:12,2026-03-25 04:01:27,Rajasthan
16706,BRW0008879,Priya Mehta,9.064240e+09,user19316@example.com,Pune,2026-03-08 01:08:17,2026-05-19 19:34:08,Maharashtra
20752,BRW0008879,Neha Singh,9.494792e+09,user4825@example.com,Jaipur,2026-02-16 02:35:43,2026-06-03 20:38:43,Rajasthan
22477,BRW0008879,Neha Singh,9.829333e+09,user17018@example.com,Delhi,2025-02-19 03:22:20,2025-12-24 11:32:33,Delhi


In [ ]:
agent_counts = data["agents"]["agent_id"].value_counts()

print("Total agent records:", len(data["agents"]))
print("Unique agent IDs:", data["agents"]["agent_id"].nunique())
print("Agent IDs appearing more than once:", (agent_counts > 1).sum())

display(agent_counts[agent_counts > 1].head(20))

Total agent records: 30000
Unique agent IDs: 1000
Agent IDs appearing more than once: 1000


,count
agent_id,
AGT0000875,48
AGT0000367,48
AGT0000533,48
AGT0000843,47
AGT0000540,46
AGT0000876,46
AGT0000563,45
AGT0000543,45
AGT0000440,45


In [ ]:
repeated_agent = agent_counts[agent_counts > 1].index[0]

print("Example repeated agent_id:", repeated_agent)

display(
    data["agents"][
        data["agents"]["agent_id"] == repeated_agent
    ]
)

Example repeated agent_id: AGT0000875


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
3236,AGT0000875,EMP00661,Priya Mehta,VND0000001,FIELD,INACTIVE,2025-06-28 02:13:31,2026-02-18 01:30:58
3340,AGT0000875,EMP00945,Rohan Patel,VND0000012,DIGITAL,SUSPENDED,2025-03-01 09:31:28,2025-11-05 12:46:51
3402,AGT0000875,EMP00007,Sneha Das,VND0000011,T2,INACTIVE,2025-03-10 22:23:24,2025-10-15 20:52:49
5091,AGT0000875,EMP00816,Rohan Patel,VND0000001,DIGITAL,INACTIVE,2024-02-03 13:08:55,2025-05-27 14:10:07
5314,AGT0000875,EMP00103,Neha Singh,VND0000004,DIGITAL,ACTIVE,2025-05-16 15:53:45,2025-07-19 23:28:54
6272,AGT0000875,EMP00615,Aarav Sharma,VND0000008,T3,INACTIVE,2025-05-20 09:00:48,2025-08-26 12:58:33
6701,AGT0000875,EMP00352,Neha Singh,VND0000009,DIGITAL,ACTIVE,2025-01-13 15:11:32,2026-06-20 16:28:50
6900,AGT0000875,EMP00176,Priya Mehta,VND0000015,FIELD,ACTIVE,2024-12-03 11:36:05,2025-11-05 20:05:59
8105,AGT0000875,EMP00570,Ananya Rao,VND0000007,T2,INACTIVE,2025-07-30 04:27:28,2025-03-20 08:49:00
9514,AGT0000875,EMP00596,Sneha Das,VND0000014,T2,ACTIVE,2025-01-12 07:30:48,2026-07-10 18:40:30


In [ ]:
print("Agent identity conflicts:",
      (agent_identity_check["unique_employee_codes"] > 1).sum())

Agent identity conflicts: 1000


In [ ]:
print(
    "Percentage of agent IDs with identity conflicts:",
    round(
        (agent_identity_check["unique_employee_codes"] > 1).sum()
        / len(agent_identity_check) * 100,
        2
    ),
    "%"
)

Percentage of agent IDs with identity conflicts: 100.0 %


In [ ]:
print(data["agents"].columns.tolist())

['agent_id', 'employee_code', 'agent_name', 'vendor_id', 'team', 'status', 'joined_at', 'updated_at']


In [ ]:
for i, col in enumerate(data["agents"].columns):
    print(i, "->", col)

0 -> agent_id
1 -> employee_code
2 -> agent_name
3 -> vendor_id
4 -> team
5 -> status
6 -> joined_at
7 -> updated_at


In [ ]:
agent_identity_check = (
    data["agents"]
    .groupby("agent_id")
    .agg(
        records=("agent_id", "size"),
        unique_employee_codes=("employee_code", "nunique"),
        unique_names=("agent_name", "nunique"),
        unique_vendors=("vendor_id", "nunique"),
        unique_teams=("team", "nunique"),
        unique_statuses=("status", "nunique")
    )
    .reset_index()
)

display(
    agent_identity_check
    .sort_values(
        ["unique_names", "unique_employee_codes"],
        ascending=False
    )
    .head(20)
)

,agent_id,records,unique_employee_codes,unique_names,unique_vendors,unique_teams,unique_statuses
366,AGT0000367,48,48,10,15,5,3
532,AGT0000533,48,47,10,15,5,3
874,AGT0000875,48,47,10,13,5,3
842,AGT0000843,47,46,10,14,5,3
539,AGT0000540,46,45,10,15,5,3
542,AGT0000543,45,45,10,15,5,3
562,AGT0000563,45,45,10,15,5,3
875,AGT0000876,46,45,10,14,4,3
55,AGT0000056,44,44,10,15,5,3
439,AGT0000440,45,44,10,13,5,3


In [ ]:
affected_agents = agent_identity_check[
    (agent_identity_check["unique_employee_codes"] > 1) |
    (agent_identity_check["unique_names"] > 1)
]

print("Total unique agent IDs:",
      data["agents"]["agent_id"].nunique())

print("Agent IDs with identity conflicts:",
      len(affected_agents))

print("Percentage affected:",
      round(
          len(affected_agents)
          / data["agents"]["agent_id"].nunique()
          * 100,
          2
      ),
      "%")

Total unique agent IDs: 1000
Agent IDs with identity conflicts: 1000
Percentage affected: 100.0 %


In [ ]:
example_agent = affected_agents.iloc[0]["agent_id"]

print("Example agent_id:", example_agent)

display(
    data["agents"][
        data["agents"]["agent_id"] == example_agent
    ].sort_values("updated_at")
)

Example agent_id: AGT0000001


,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
9554,AGT0000001,EMP00883,Sneha Das,VND0000010,T3,INACTIVE,2025-09-15 12:18:43,2025-02-08 10:18:20
3016,AGT0000001,EMP00285,Sneha Das,VND0000013,DIGITAL,INACTIVE,2025-05-02 11:13:31,2025-02-14 12:38:08
18301,AGT0000001,EMP00191,Priya Mehta,VND0000015,FIELD,ACTIVE,2025-11-04 07:01:56,2025-03-09 16:24:54
21572,AGT0000001,EMP00745,Vikram Shah,VND0000008,T3,ACTIVE,2024-02-08 16:05:41,2025-03-09 22:26:39
8967,AGT0000001,EMP01082,Neha Singh,VND0000013,T1,INACTIVE,2025-05-02 01:36:54,2025-04-25 02:56:44
6520,AGT0000001,EMP00632,Ananya Rao,VND0000007,DIGITAL,ACTIVE,2024-08-02 00:51:56,2025-08-30 06:00:31
19963,AGT0000001,EMP00903,Sneha Das,VND0000002,T3,INACTIVE,2024-02-21 17:11:06,2025-09-18 08:17:45
9420,AGT0000001,EMP00723,Rahul Verma,VND0000013,T2,ACTIVE,2025-03-04 14:23:32,2025-10-11 06:30:44
25883,AGT0000001,EMP00404,Ananya Rao,VND0000014,T2,SUSPENDED,2025-11-13 07:54:11,2025-10-27 07:59:16
15127,AGT0000001,EMP00227,Sneha Das,VND0000007,T3,INACTIVE,2024-12-06 13:45:28,2025-11-26 03:28:09


In [ ]:
borrower_identity_check = (
    data["borrowers"]
    .groupby("borrower_id")
    .agg(
        records=("borrower_id", "size"),
        unique_names=("name", "nunique"),
        unique_phones=("phone", "nunique"),
        unique_emails=("email", "nunique"),
        unique_cities=("city", "nunique")
    )
    .reset_index()
)

display(
    borrower_identity_check
    .sort_values(
        ["unique_names", "unique_phones"],
        ascending=False
    )
    .head(20)
)

,borrower_id,records,unique_names,unique_phones,unique_emails,unique_cities
1153,BRW0001237,10,9,10,10,7
2217,BRW0002398,9,8,9,9,6
4624,BRW0005021,9,8,9,6,7
8150,BRW0008879,11,7,11,10,8
2327,BRW0002520,10,7,10,10,5
4966,BRW0005385,10,7,10,10,5
5812,BRW0006302,11,7,10,10,7
10144,BRW0011052,9,7,9,9,6
3154,BRW0003415,8,7,8,8,6
3731,BRW0004044,9,7,8,8,4


In [ ]:
affected_borrower_ids = borrower_identity_check[
    (borrower_identity_check["unique_names"] > 1) |
    (borrower_identity_check["unique_phones"] > 1) |
    (borrower_identity_check["unique_emails"] > 1)
]

print(
    "Total unique borrower IDs:",
    data["borrowers"]["borrower_id"].nunique()
)

print(
    "Borrower IDs with identity conflicts:",
    len(affected_borrower_ids)
)

print(
    "Percentage affected:",
    round(
        len(affected_borrower_ids)
        / data["borrowers"]["borrower_id"].nunique()
        * 100,
        2
    ),
    "%"
)

Total unique borrower IDs: 11015
Borrower IDs with identity conflicts: 8518
Percentage affected: 77.33 %


In [ ]:
account_borrower_check = (
    data["accounts"]
    .groupby("account_id")
    .agg(
        records=("account_id", "size"),
        unique_borrowers=("borrower_id", "nunique")
    )
    .reset_index()
)

display(
    account_borrower_check
    .sort_values(
        ["records", "unique_borrowers"],
        ascending=False
    )
    .head(20)
)

,account_id,records,unique_borrowers
0,ACC0000001,1,1
1,ACC0000002,1,1
2,ACC0000003,1,1
3,ACC0000004,1,1
4,ACC0000005,1,1
6,ACC0000007,1,1
7,ACC0000008,1,1
8,ACC0000009,1,1
9,ACC0000010,1,1
10,ACC0000011,1,1


In [ ]:
borrower_account_counts = (
    data["accounts"]
    .groupby("borrower_id")
    .size()
    .reset_index(name="account_count")
)

print(
    "Total borrowers appearing in accounts:",
    borrower_account_counts["borrower_id"].nunique()
)

print(
    "Maximum accounts for one borrower:",
    borrower_account_counts["account_count"].max()
)

display(
    borrower_account_counts
    .sort_values("account_count", ascending=False)
    .head(20)
)

Total borrowers appearing in accounts: 10943
Maximum accounts for one borrower: 11


,borrower_id,account_count
4661,BRW0005106,11
738,BRW0000800,10
8511,BRW0009333,10
5330,BRW0005835,10
4738,BRW0005190,9
2876,BRW0003162,9
6481,BRW0007121,9
1597,BRW0001748,9
1850,BRW0002034,9
4050,BRW0004449,9


In [ ]:
# STEP 18 - Find date/time related columns in every dataset

timestamp_columns = []

for dataset_name, df in data.items():
    for col in df.columns:
        if any(word in col.lower() for word in [
            "at", "date", "time", "timestamp"
        ]):
            timestamp_columns.append({
                "Dataset": dataset_name,
                "Column": col
            })

timestamp_df = pd.DataFrame(timestamp_columns)

display(timestamp_df)

,Dataset,Column
0,call_dispositions,event_at
1,sms_events,event_at
2,sms_events,template_code
3,payments,event_at
4,payments,payment_status
5,accounts,status
6,accounts,opened_at
7,accounts,timezone
8,agent_sessions,login_at
9,agent_sessions,timezone


In [ ]:
timestamp_quality = []

for dataset_name, df in data.items():

    for col in df.columns:

        if any(word in col.lower() for word in [
            "at", "date", "time", "timestamp"
        ]):

            total = len(df)
            missing = df[col].isna().sum()

            timestamp_quality.append({
                "Dataset": dataset_name,
                "Column": col,
                "Total Rows": total,
                "Missing": missing,
                "Missing %": round((missing / total) * 100, 2)
            })

timestamp_quality_df = pd.DataFrame(timestamp_quality)

display(
    timestamp_quality_df
    .sort_values("Missing %", ascending=False)
)

,Dataset,Column,Total Rows,Missing,Missing %
16,field_visits,scheduled_at,25000,250,1.0
0,call_dispositions,event_at,35000,0,0.0
2,sms_events,template_code,45000,0,0.0
1,sms_events,event_at,45000,0,0.0
4,payments,payment_status,25500,0,0.0
5,accounts,status,30000,0,0.0
6,accounts,opened_at,30000,0,0.0
3,payments,event_at,25500,0,0.0
8,agent_sessions,login_at,15000,0,0.0
9,agent_sessions,timezone,15000,0,0.0


In [ ]:
# STEP 18A - Identify actual timestamp columns

timestamp_columns = []

for dataset_name, df in data.items():
    for col in df.columns:
        col_lower = col.lower()

        if (
            col_lower.endswith("_at")
            or "timestamp" in col_lower
            or col_lower in ["date", "time"]
        ):
            timestamp_columns.append({
                "Dataset": dataset_name,
                "Column": col
            })

timestamp_df = pd.DataFrame(timestamp_columns)

display(timestamp_df)

,Dataset,Column
0,call_dispositions,event_at
1,sms_events,event_at
2,payments,event_at
3,accounts,opened_at
4,agent_sessions,login_at
5,agent_sessions,logout_at
6,account_status_history,event_at
7,account_status_history,recorded_at
8,field_visits,event_at
9,field_visits,scheduled_at


In [ ]:
# STEP 18B - Timestamp missing-value audit

timestamp_quality = []

for dataset_name, df in data.items():

    for col in df.columns:
        col_lower = col.lower()

        if (
            col_lower.endswith("_at")
            or "timestamp" in col_lower
            or col_lower in ["date", "time"]
        ):

            total = len(df)
            missing = df[col].isna().sum()

            timestamp_quality.append({
                "Dataset": dataset_name,
                "Column": col,
                "Total Rows": total,
                "Missing": missing,
                "Missing %": round((missing / total) * 100, 2)
            })

timestamp_quality_df = pd.DataFrame(timestamp_quality)

display(
    timestamp_quality_df
    .sort_values("Missing %", ascending=False)
)

,Dataset,Column,Total Rows,Missing,Missing %
9,field_visits,scheduled_at,25000,250,1.0
0,call_dispositions,event_at,35000,0,0.0
2,payments,event_at,25500,0,0.0
3,accounts,opened_at,30000,0,0.0
4,agent_sessions,login_at,15000,0,0.0
1,sms_events,event_at,45000,0,0.0
5,agent_sessions,logout_at,15000,0,0.0
6,account_status_history,event_at,60000,0,0.0
7,account_status_history,recorded_at,60000,0,0.0
8,field_visits,event_at,25000,0,0.0


In [ ]:
timestamp_types = []

for dataset_name, df in data.items():

    for col in df.columns:
        col_lower = col.lower()

        if (
            col_lower.endswith("_at")
            or "timestamp" in col_lower
            or col_lower in ["date", "time"]
        ):

            timestamp_types.append({
                "Dataset": dataset_name,
                "Column": col,
                "Data Type": str(df[col].dtype)
            })

timestamp_types_df = pd.DataFrame(timestamp_types)

display(timestamp_types_df)

,Dataset,Column,Data Type
0,call_dispositions,event_at,object
1,sms_events,event_at,object
2,payments,event_at,object
3,accounts,opened_at,object
4,agent_sessions,login_at,object
5,agent_sessions,logout_at,object
6,account_status_history,event_at,object
7,account_status_history,recorded_at,object
8,field_visits,event_at,object
9,field_visits,scheduled_at,object


In [ ]:
for dataset_name, df in data.items():

    for col in df.columns:
        col_lower = col.lower()

        if (
            col_lower.endswith("_at")
            or "timestamp" in col_lower
        ):

            print("\n" + "=" * 60)
            print("DATASET:", dataset_name)
            print("COLUMN:", col)

            display(df[[col]].head(5))


DATASET: call_dispositions
COLUMN: event_at


,event_at
0,2026-03-07 17:30:48
1,2026-04-04 11:59:50
2,2026-03-11 01:25:13
3,2026-03-17 14:49:43
4,2026-03-07 06:07:26



DATASET: sms_events
COLUMN: event_at


,event_at
0,2026-04-24 20:57:06
1,2026-05-24 11:28:20
2,2026-03-05 20:23:20
3,2026-05-11 04:56:15
4,2026-03-24 08:41:53



DATASET: payments
COLUMN: event_at


...

,end_at
0,2026-04-24 06:56:01
1,2026-05-18 17:51:31
2,2026-05-18 16:02:51
3,2026-05-29 06:15:30
4,2026-07-08 04:16:21



DATASET: whatsapp_events
COLUMN: event_at


,event_at
0,2026-02-27 17:49:38
1,2026-02-09 11:55:18
2,2026-07-08 05:16:13
3,2026-05-20 19:48:52
4,2026-01-10 10:37:24



DATASET: promises_to_pay
COLUMN: event_at


,event_at
0,2026-04-18 21:11:02
1,2026-07-08 21:49:28
2,2026-08-08 08:57:34
3,2026-01-11 06:04:20
4,2026-05-08 01:42:07


In [ ]:
timezone_columns = []

for dataset_name, df in data.items():
    for col in df.columns:
        if "timezone" in col.lower() or "tz" in col.lower():
            timezone_columns.append({
                "Dataset": dataset_name,
                "Column": col
            })

timezone_df = pd.DataFrame(timezone_columns)

display(timezone_df)

,Dataset,Column
0,accounts,timezone
1,agent_sessions,timezone
2,calls,timezone
3,vendor_telephony,timezone


In [ ]:
for dataset_name, df in data.items():

    for col in df.columns:

        if "timezone" in col.lower() or "tz" in col.lower():

            print("\n" + "=" * 60)
            print("DATASET:", dataset_name)
            print("COLUMN:", col)

            display(df[col].value_counts(dropna=False))


DATASET: accounts
COLUMN: timezone


,count
timezone,
UTC,10096
Asia/Kolkata,9981
Asia/Dubai,9923



DATASET: agent_sessions
COLUMN: timezone


,count
timezone,
Asia/Kolkata,7506
UTC,7494



DATASET: calls
COLUMN: timezone


,count
timezone,
Asia/Kolkata,30485
Asia/Dubai,30464
UTC,30401



DATASET: vendor_telephony
COLUMN: timezone


,count
timezone,
UTC,8
Asia/Kolkata,7


In [ ]:
# STEP: Check timestamp conflicts across related datasets

print("=== TIMESTAMP CONFLICT CHECK ===")

# Accounts: opened_at
accounts_time = data["accounts"][["account_id", "opened_at", "timezone"]].copy()

# Convert opened_at to datetime
accounts_time["opened_at"] = pd.to_datetime(
    accounts_time["opened_at"],
    errors="coerce"
)

print("\nAccounts timestamp range:")
print("Minimum:", accounts_time["opened_at"].min())
print("Maximum:", accounts_time["opened_at"].max())

# Agent sessions: login and logout
sessions_time = data["agent_sessions"][
    ["session_id", "login_at", "logout_at", "timezone"]
].copy()

sessions_time["login_at"] = pd.to_datetime(
    sessions_time["login_at"],
    errors="coerce"
)

sessions_time["logout_at"] = pd.to_datetime(
    sessions_time["logout_at"],
    errors="coerce"
)

# Check sessions where logout happens before login
invalid_sessions = sessions_time[
    sessions_time["logout_at"] < sessions_time["login_at"]
]

print("\nAgent sessions with logout before login:")
print(len(invalid_sessions))

display(invalid_sessions.head(10))

=== TIMESTAMP CONFLICT CHECK ===

Accounts timestamp range:
Minimum: 2024-01-01 00:02:27
Maximum: 2025-11-30 23:52:36

Agent sessions with logout before login:
0


,session_id,login_at,logout_at,timezone


In [ ]:
# STEP: Check duplicate payment references

print("=== DUPLICATE PAYMENT REFERENCES ===")

payments = data["payments"]

duplicate_payment_refs = (
    payments.groupby("payment_reference")
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

# Keep only references appearing more than once
duplicate_payment_refs = duplicate_payment_refs[
    duplicate_payment_refs["records"] > 1
]

print("Number of duplicated payment references:",
      len(duplicate_payment_refs))

display(duplicate_payment_refs.head(20))


=== DUPLICATE PAYMENT REFERENCES ===
Number of duplicated payment references: 3745


,payment_reference,records
19524,TXN0000065723,5
1993,TXN0000006936,5
14862,TXN0000050468,5
585,TXN0000002005,5
6280,TXN0000021482,5
13081,TXN0000044312,5
15513,TXN0000052559,5
10973,TXN0000037243,4
4035,TXN0000013902,4
17293,TXN0000058512,4


In [ ]:
# STEP: Inspect one duplicated payment reference

example_payment_ref = duplicate_payment_refs.iloc[0]["payment_reference"]

print("Example payment_reference:", example_payment_ref)

display(
    data["payments"][
        data["payments"]["payment_reference"] == example_payment_ref
    ]
)

Example payment_reference: TXN0000065723


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
3610,PAYMENT0003611,ACC0000415,BRW0010475,2026-06-25 07:09:53,TXN0000065723,88993.39,PENDING,UPI,VND0000005
3997,PAYMENT0003998,ACC0028192,BRW0011219,2026-04-20 15:26:26,TXN0000065723,10078.02,SUCCESS,UPI,VND0000010
5114,PAYMENT0005115,ACC0028014,BRW0004112,2026-05-27 02:53:34,TXN0000065723,110097.77,SUCCESS,UPI,VND0000012
12174,PAYMENT0012175,ACC0005539,BRW0009713,2026-06-30 00:18:26,TXN0000065723,123802.94,FAILED,NACH,VND0000005
25229,PAYMENT0005115,ACC0028014,BRW0004112,2026-05-27 02:53:34,TXN0000065723,110097.77,SUCCESS,UPI,VND0000012


In [ ]:
# STEP: Find exact duplicate payment records

payments = data["payments"]

exact_duplicate_payments = payments[
    payments.duplicated(keep=False)
].sort_values("payment_reference")

print("Total exact duplicate payment rows:",
      len(exact_duplicate_payments))

display(exact_duplicate_payments.head(20))

Total exact duplicate payment rows: 972


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
25010,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
23069,PAYMENT0023070,ACC0007158,BRW0002020,2026-01-08 00:16:00,TXN0000000314,120410.28,SUCCESS,UPI,VND0000009
25058,PAYMENT0023070,ACC0007158,BRW0002020,2026-01-08 00:16:00,TXN0000000314,120410.28,SUCCESS,UPI,VND0000009
11613,PAYMENT0011614,ACC0003825,BRW0004986,2026-01-01 15:42:05,TXN0000000466,35186.17,PENDING,CARD,VND0000010
25477,PAYMENT0011614,ACC0003825,BRW0004986,2026-01-01 15:42:05,TXN0000000466,35186.17,PENDING,CARD,VND0000010
25023,PAYMENT0012446,ACC0003910,BRW0009459,2026-06-09 11:35:03,TXN0000000475,133757.22,SUCCESS,UPI,VND0000004
12445,PAYMENT0012446,ACC0003910,BRW0009459,2026-06-09 11:35:03,TXN0000000475,133757.22,SUCCESS,UPI,VND0000004
25325,PAYMENT0011853,ACC0009042,BRW0011298,2026-04-02 05:45:28,TXN0000000512,117341.22,SUCCESS,CASH,VND0000007
11852,PAYMENT0011853,ACC0009042,BRW0011298,2026-04-02 05:45:28,TXN0000000512,117341.22,SUCCESS,CASH,VND0000007


In [ ]:
# STEP: Measure impact of exact duplicate removal

total_payment_rows = len(data["payments"])

unique_payment_rows = len(
    data["payments"].drop_duplicates()
)

duplicate_rows = total_payment_rows - unique_payment_rows

print("Total payment rows:", total_payment_rows)
print("Unique payment rows:", unique_payment_rows)
print("Exact duplicate rows:", duplicate_rows)

print(
    "Percentage of rows duplicated:",
    round(duplicate_rows / total_payment_rows * 100, 2),
    "%"
)

Total payment rows: 25500
Unique payment rows: 25014
Exact duplicate rows: 486
Percentage of rows duplicated: 1.91 %


In [ ]:
# STEP: Find payment references linked to multiple payment IDs

payment_reference_conflicts = (
    data["payments"]
    .groupby("payment_reference")
    .agg(
        payment_records=("payment_id", "size"),
        unique_payment_ids=("payment_id", "nunique"),
        unique_accounts=("account_id", "nunique"),
        unique_borrowers=("borrower_id", "nunique"),
        unique_amounts=("amount", "nunique"),
        unique_statuses=("payment_status", "nunique")
    )
    .reset_index()
)

# Keep references connected to more than one payment ID
payment_reference_conflicts = payment_reference_conflicts[
    payment_reference_conflicts["unique_payment_ids"] > 1
]

payment_reference_conflicts = payment_reference_conflicts.sort_values(
    "unique_payment_ids",
    ascending=False
)

print(
    "Payment references linked to multiple payment IDs:",
    len(payment_reference_conflicts)
)

display(payment_reference_conflicts.head(20))

Payment references linked to multiple payment IDs: 3407


,payment_reference,payment_records,unique_payment_ids,unique_accounts,unique_borrowers,unique_amounts,unique_statuses
13081,TXN0000044312,5,5,5,5,5,2
9732,TXN0000033038,4,4,4,4,4,3
13088,TXN0000044341,4,4,4,4,4,3
15513,TXN0000052559,5,4,4,4,4,3
18461,TXN0000062318,4,4,4,4,4,3
15283,TXN0000051851,4,4,4,4,4,1
11022,TXN0000037402,4,4,4,4,4,3
18924,TXN0000063848,4,4,4,4,4,1
8205,TXN0000027954,4,4,4,4,4,2
17657,TXN0000059686,4,4,4,4,4,3


In [ ]:
# STEP: Inspect a conflicting payment reference

example_conflict_ref = payment_reference_conflicts.iloc[0]["payment_reference"]

print("Example conflicting payment_reference:", example_conflict_ref)

display(
    data["payments"][
        data["payments"]["payment_reference"] == example_conflict_ref
    ].sort_values("event_at")
)

Example conflicting payment_reference: TXN0000044312


,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
22912,PAYMENT0022913,ACC0025320,BRW0008750,2026-01-02 14:34:45,TXN0000044312,125333.52,SUCCESS,NETBANKING,VND0000007
12956,PAYMENT0012957,ACC0025419,BRW0006475,2026-03-17 20:29:49,TXN0000044312,4943.80,SUCCESS,CARD,VND0000015
15119,PAYMENT0015120,ACC0008230,BRW0004237,2026-03-19 06:17:38,TXN0000044312,82139.32,SUCCESS,NACH,VND0000011
11734,PAYMENT0011735,ACC0021560,BRW0005868,2026-05-11 00:30:09,TXN0000044312,61143.84,FAILED,UPI,VND0000011
19781,PAYMENT0019782,ACC0008440,BRW0001094,2026-07-30 14:18:01,TXN0000044312,98129.03,SUCCESS,NETBANKING,VND0000011


In [ ]:
# STEP: Measure payment reference conflicts

total_references = data["payments"]["payment_reference"].nunique()

conflicting_references = len(payment_reference_conflicts)

print("Total unique payment references:", total_references)
print("Conflicting payment references:", conflicting_references)

print(
    "Percentage of references with conflicts:",
    round(conflicting_references / total_references * 100, 2),
    "%"
)

Total unique payment references: 20821
Conflicting payment references: 3407
Percentage of references with conflicts: 16.36 %


In [ ]:


golden = {}

for name, df in data.items():
    golden[name] = df.copy()

print("Datasets copied into Golden Dataset workspace:")
print("-" * 50)

for name, df in golden.items():
    print(f"{name:25s} {df.shape[0]:>8,} rows  |  {df.shape[1]:>3} columns")

Datasets copied into Golden Dataset workspace:
--------------------------------------------------
call_dispositions           35,000 rows  |    8 columns
sms_events                  45,000 rows  |    8 columns
payments                    25,500 rows  |    9 columns
accounts                    30,000 rows  |   11 columns
agent_sessions              15,000 rows  |    7 columns
account_status_history      60,000 rows  |    8 columns
field_visits                25,000 rows  |   10 columns
call_attempts              120,000 rows  |    9 columns
calls                       91,350 rows  |   11 columns
agents                      30,000 rows  |    8 columns
complaints                   8,000 rows  |    9 columns
borrowers                   30,600 rows  |    8 columns
campaigns                      120 rows  |    7 columns
vendor_telephony                15 rows  |    6 columns
daily_targeting             45,000 rows  |    7 columns
whatsapp_events             60,600 rows  |    8 columns
data_d

In [ ]:


quality_summary = []

for name, df in golden.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Duplicate %": round(df.duplicated().mean() * 100, 2)
    })

quality_df = pd.DataFrame(quality_summary)

display(
    quality_df.sort_values(
        "Missing Values",
        ascending=False
    )
)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Duplicate %
7,call_attempts,120000,9,2400,0,0.00
8,calls,91350,11,1827,1271,1.39
11,borrowers,30600,8,1509,600,1.96
3,accounts,30000,11,455,0,0.00
2,payments,25500,9,382,486,1.91
6,field_visits,25000,10,250,0,0.00
1,sms_events,45000,8,0,0,0.00
0,call_dispositions,35000,8,0,0,0.00
5,account_status_history,60000,8,0,0,0.00
4,agent_sessions,15000,7,0,0,0.00


In [ ]:
missing_details = []

for name, df in golden.items():
    for col in df.columns:
        missing = int(df[col].isna().sum())

        if missing > 0:
            missing_details.append({
                "Dataset": name,
                "Column": col,
                "Missing Values": missing,
                "Missing %": round((missing / len(df)) * 100, 2)
            })

missing_df = pd.DataFrame(missing_details)

display(
    missing_df.sort_values(
        "Missing %",
        ascending=False
    )
)

,Dataset,Column,Missing Values,Missing %
6,borrowers,email,895,2.92
5,borrowers,phone,614,2.01
4,calls,agent_id,1827,2.00
3,call_attempts,vendor_id,2400,2.00
1,accounts,borrower_id,455,1.52
0,payments,payment_reference,382,1.50
2,field_visits,scheduled_at,250,1.00


In [ ]:
duplicate_details = []

for name, df in golden.items():
    duplicate_count = int(df.duplicated().sum())

    if duplicate_count > 0:
        duplicate_details.append({
            "Dataset": name,
            "Total Rows": len(df),
            "Duplicate Rows": duplicate_count,
            "Duplicate %": round((duplicate_count / len(df)) * 100, 2)
        })

duplicate_df = pd.DataFrame(duplicate_details)

display(
    duplicate_df.sort_values(
        "Duplicate %",
        ascending=False
    )
)

,Dataset,Total Rows,Duplicate Rows,Duplicate %
2,borrowers,30600,600,1.96
0,payments,25500,486,1.91
1,calls,91350,1271,1.39
3,whatsapp_events,60600,600,0.99


In [ ]:
duplicate_key_details = []

key_columns = {
    "borrowers": ["borrower_id"],
    "payments": ["payment_id", "payment_reference"],
    "calls": ["call_id"],
    "whatsapp_events": ["whatsapp_event_id"]
}

for dataset, columns in key_columns.items():
    df = golden[dataset]

    for col in columns:
        if col in df.columns:
            duplicate_count = int(df[col].duplicated().sum())

            duplicate_key_details.append({
                "Dataset": dataset,
                "Column": col,
                "Duplicate IDs": duplicate_count,
                "Unique IDs": int(df[col].nunique()),
                "Total Rows": len(df),
                "Duplicate %": round(
                    duplicate_count / len(df) * 100, 2
                )
            })

duplicate_key_df = pd.DataFrame(duplicate_key_details)

display(
    duplicate_key_df.sort_values(
        "Duplicate %",
        ascending=False
    )
)

,Dataset,Column,Duplicate IDs,Unique IDs,Total Rows,Duplicate %
0,borrowers,borrower_id,19585,11015,30600,64.00
2,payments,payment_reference,4678,20821,25500,18.35
1,payments,payment_id,500,25000,25500,1.96
3,calls,call_id,1350,90000,91350,1.48
4,whatsapp_events,whatsapp_event_id,600,60000,60600,0.99


In [ ]:

borrower_id_counts = (
    golden["borrowers"]["borrower_id"]
    .value_counts()
    .reset_index()
)

borrower_id_counts.columns = ["borrower_id", "Record Count"]

print("Total unique borrower IDs:",
      golden["borrowers"]["borrower_id"].nunique())

print("Total borrower records:",
      len(golden["borrowers"]))

print("Borrower IDs appearing more than once:",
      (borrower_id_counts["Record Count"] > 1).sum())

display(
    borrower_id_counts[
        borrower_id_counts["Record Count"] > 1
    ].head(20)
)

Total unique borrower IDs: 11015
Total borrower records: 30600
Borrower IDs appearing more than once: 8566


,borrower_id,Record Count
0,BRW0008879,11
1,BRW0006302,11
2,BRW0002520,10
3,BRW0001237,10
4,BRW0005385,10
5,BRW0001843,9
6,BRW0003622,9
7,BRW0005630,9
8,BRW0002013,9
9,BRW0004843,9


In [ ]:


borrower_identity_check = (
    golden["borrowers"]
    .groupby("borrower_id")
    .agg(
        records=("borrower_id", "size"),
        unique_names=("name", "nunique"),
        unique_phones=("phone", "nunique"),
        unique_emails=("email", "nunique"),
        unique_cities=("city", "nunique")
    )
    .reset_index()
)

display(
    borrower_identity_check
    .sort_values(
        ["unique_names", "unique_phones", "unique_emails"],
        ascending=False
    )
    .head(20)
)

,borrower_id,records,unique_names,unique_phones,unique_emails,unique_cities
1153,BRW0001237,10,9,10,10,7
2217,BRW0002398,9,8,9,9,6
4624,BRW0005021,9,8,9,6,7
8150,BRW0008879,11,7,11,10,8
2327,BRW0002520,10,7,10,10,5
4966,BRW0005385,10,7,10,10,5
5812,BRW0006302,11,7,10,10,7
10144,BRW0011052,9,7,9,9,6
3154,BRW0003415,8,7,8,8,6
3731,BRW0004044,9,7,8,8,4


In [ ]:


borrower_conflicts = borrower_identity_check[
    (borrower_identity_check["unique_names"] > 1) |
    (borrower_identity_check["unique_phones"] > 1) |
    (borrower_identity_check["unique_emails"] > 1) |
    (borrower_identity_check["unique_cities"] > 1)
]

total_unique_borrowers = data["borrowers"]["borrower_id"].nunique()
conflicting_borrowers = len(borrower_conflicts)

print("Total unique borrower IDs:", total_unique_borrowers)
print("Borrower IDs with identity conflicts:", conflicting_borrowers)

print(
    "Percentage of borrower IDs with identity conflicts:",
    round(conflicting_borrowers / total_unique_borrowers * 100, 2),
    "%"
)

Total unique borrower IDs: 11015
Borrower IDs with identity conflicts: 8518
Percentage of borrower IDs with identity conflicts: 77.33 %


In [ ]:
print("=" * 70)
print("FINAL DATA QUALITY ASSESSMENT")
print("=" * 70)

print("\n1. DATASET OVERVIEW")
print("-" * 70)

for name, df in golden.items():
    print(f"{name:25s} | {len(df):>8,} rows | {len(df.columns):>2} columns")

print("\n2. MISSING VALUES")
print("-" * 70)

display(
    missing_df.sort_values(
        "Missing %",
        ascending=False
    )
)

print("\n3. DUPLICATE RECORDS")
print("-" * 70)

display(
    duplicate_df.sort_values(
        "Duplicate %",
        ascending=False
    )
)

print("\n4. DUPLICATE KEY ANALYSIS")
print("-" * 70)

display(
    duplicate_key_df.sort_values(
        "Duplicate %",
        ascending=False
    )
)

print("\n5. AGENT IDENTITY CONFLICTS")
print("-" * 70)

print("Total unique agent IDs:",
      data["agents"]["agent_id"].nunique())

print("Agent IDs with identity conflicts:",
      len(affected_agents))

print(
    "Percentage of agent IDs with identity conflicts:",
    round(
        len(affected_agents) /
        data["agents"]["agent_id"].nunique() * 100,
        2
    ),
    "%"
)

print("\n6. BORROWER IDENTITY CONFLICTS")
print("-" * 70)

print("Total unique borrower IDs:",
      data["borrowers"]["borrower_id"].nunique())

print("Borrower IDs with identity conflicts:",
      len(borrower_conflicts))

print(
    "Percentage of borrower IDs with identity conflicts:",
    round(
        len(borrower_conflicts) /
        data["borrowers"]["borrower_id"].nunique() * 100,
        2
    ),
    "%"
)

print("\n7. PAYMENT REFERENCE CONFLICTS")
print("-" * 70)

print("Total unique payment references:",
      total_references)

print("Conflicting payment references:",
      conflicting_references)

print(
    "Percentage of payment references with conflicts:",
    round(
        conflicting_references /
        total_references * 100,
        2
    ),
    "%"
)

print("\n8. TIMESTAMP QUALITY")
print("-" * 70)

print("Agent sessions with logout before login:",
      len(invalid_sessions))

print("\n9. EXACT PAYMENT DUPLICATES")
print("-" * 70)

print("Total payment rows:",
      total_payment_rows)

print("Exact duplicate payment rows:",
      duplicate_rows)

print(
    "Percentage of payment rows duplicated:",
    round(
        duplicate_rows /
        total_payment_rows * 100,
        2
    ),
    "%"
)

print("\n" + "=" * 70)
print("DATA QUALITY ASSESSMENT COMPLETE")
print("=" * 70)

FINAL DATA QUALITY ASSESSMENT

1. DATASET OVERVIEW
----------------------------------------------------------------------
call_dispositions         |   35,000 rows |  8 columns
sms_events                |   45,000 rows |  8 columns
payments                  |   25,500 rows |  9 columns
accounts                  |   30,000 rows | 11 columns
agent_sessions            |   15,000 rows |  7 columns
account_status_history    |   60,000 rows |  8 columns
field_visits              |   25,000 rows | 10 columns
call_attempts             |  120,000 rows |  9 columns
calls                     |   91,350 rows | 11 columns
agents                    |   30,000 rows |  8 columns
complaints                |    8,000 rows |  9 columns
borrowers                 |   30,600 rows |  8 columns
campaigns                 |      120 rows |  7 columns
vendor_telephony          |       15 rows |  6 columns
daily_targeting           |   45,000 rows |  7 columns
whatsapp_events           |   60,600 rows |  8 column

,Dataset,Column,Missing Values,Missing %
6,borrowers,email,895,2.92
5,borrowers,phone,614,2.01
4,calls,agent_id,1827,2.00
3,call_attempts,vendor_id,2400,2.00
1,accounts,borrower_id,455,1.52
0,payments,payment_reference,382,1.50
2,field_visits,scheduled_at,250,1.00



3. DUPLICATE RECORDS
----------------------------------------------------------------------


,Dataset,Total Rows,Duplicate Rows,Duplicate %
2,borrowers,30600,600,1.96
0,payments,25500,486,1.91
1,calls,91350,1271,1.39
3,whatsapp_events,60600,600,0.99



4. DUPLICATE KEY ANALYSIS
----------------------------------------------------------------------


,Dataset,Column,Duplicate IDs,Unique IDs,Total Rows,Duplicate %
0,borrowers,borrower_id,19585,11015,30600,64.00
2,payments,payment_reference,4678,20821,25500,18.35
1,payments,payment_id,500,25000,25500,1.96
3,calls,call_id,1350,90000,91350,1.48
4,whatsapp_events,whatsapp_event_id,600,60000,60600,0.99



5. AGENT IDENTITY CONFLICTS
----------------------------------------------------------------------
Total unique agent IDs: 1000
Agent IDs with identity conflicts: 1000
Percentage of agent IDs with identity conflicts: 100.0 %

6. BORROWER IDENTITY CONFLICTS
----------------------------------------------------------------------
Total unique borrower IDs: 11015
Borrower IDs with identity conflicts: 8518
Percentage of borrower IDs with identity conflicts: 77.33 %

7. PAYMENT REFERENCE CONFLICTS
----------------------------------------------------------------------
Total unique payment references: 20821
Conflicting payment references: 3407
Percentage of payment references with conflicts: 16.36 %

8. TIMESTAMP QUALITY
----------------------------------------------------------------------
Agent sessions with logout before login: 0

9. EXACT PAYMENT DUPLICATES
----------------------------------------------------------------------
Total payment rows: 25500
Exact duplicate payment rows: 486
Per

In [ ]:
print("=== DATASET COLUMNS ===")

for name in [
    "accounts",
    "payments",
    "calls",
    "call_attempts",
    "campaigns",
    "daily_targeting"
]:
    print(f"\n{name}:")
    print(list(data[name].columns))

=== DATASET COLUMNS ===

accounts:
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

payments:
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

calls:
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

call_attempts:
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'attempt_no', 'vendor_id', 'attempt_status']

campaigns:
['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

daily_targeting:
['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']


In [ ]:
print("========== ACCOUNTS ==========")
for c in data["accounts"].columns:
    print(c)

print("\n========== PAYMENTS ==========")
for c in data["payments"].columns:
    print(c)

print("\n========== CALLS ==========")
for c in data["calls"].columns:
    print(c)

print("\n========== CALL ATTEMPTS ==========")
for c in data["call_attempts"].columns:
    print(c)

print("\n========== CAMPAIGNS ==========")
for c in data["campaigns"].columns:
    print(c)

print("\n========== DAILY TARGETING ==========")
for c in data["daily_targeting"].columns:
    print(c)

========== ACCOUNTS ==========
account_id
borrower_id
loan_type
principal_amount
outstanding_amount
dpd
risk_segment
status
opened_at
timezone
schema_version

========== PAYMENTS ==========
payment_id
account_id
borrower_id
event_at
payment_reference
amount
payment_status
payment_method
provider_id

========== CALLS ==========
call_id
account_id
borrower_id
event_at
agent_id
campaign_id
direction
vendor_id
call_status
duration_sec
timezone

========== CALL ATTEMPTS ==========
attempt_id
account_id
borrower_id
event_at
call_id
agent_id
attempt_no
vendor_id
attempt_status

========== CAMPAIGNS ==========
campaign_id
campaign_name
channel
strategy_version
start_at
target_definition
end_at

========== DAILY TARGETING ==========
target_id
account_id
campaign_id
target_date
priority
recommended_channel
status


In [ ]:
for name in [
    "accounts",
    "payments",
    "calls",
    "call_attempts",
    "campaigns",
    "daily_targeting"
]:
    print("\n" + "=" * 50)
    print(name.upper())
    print("=" * 50)
    print(data[name].columns.tolist())


ACCOUNTS
['account_id', 'borrower_id', 'loan_type', 'principal_amount', 'outstanding_amount', 'dpd', 'risk_segment', 'status', 'opened_at', 'timezone', 'schema_version']

PAYMENTS
['payment_id', 'account_id', 'borrower_id', 'event_at', 'payment_reference', 'amount', 'payment_status', 'payment_method', 'provider_id']

CALLS
['call_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'campaign_id', 'direction', 'vendor_id', 'call_status', 'duration_sec', 'timezone']

CALL_ATTEMPTS
['attempt_id', 'account_id', 'borrower_id', 'event_at', 'call_id', 'agent_id', 'attempt_no', 'vendor_id', 'attempt_status']

CAMPAIGNS
['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

DAILY_TARGETING
['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']


In [ ]:

print("=" * 70)
print("BUSINESS PERFORMANCE BASELINE")
print("=" * 70)

# 1. ACCOUNT PORTFOLIO
accounts = data["accounts"].copy()

print("\n1. ACCOUNT PORTFOLIO")
print("-" * 50)

print("Total accounts:", accounts["account_id"].nunique())
print("Total borrowers:", accounts["borrower_id"].nunique())

if "principal_amount" in accounts.columns:
    print(
        "Total principal amount:",
        round(pd.to_numeric(accounts["principal_amount"], errors="coerce").sum(), 2)
    )

if "outstanding_amount" in accounts.columns:
    print(
        "Total outstanding amount:",
        round(pd.to_numeric(accounts["outstanding_amount"], errors="coerce").sum(), 2)
    )

if "dpd" in accounts.columns:
    print("\nDPD distribution:")
    display(
        accounts["dpd"]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Accounts")
    )

if "risk_segment" in accounts.columns:
    print("\nRisk segment distribution:")
    display(
        accounts["risk_segment"]
        .value_counts(dropna=False)
        .to_frame("Accounts")
    )

if "status" in accounts.columns:
    print("\nAccount status distribution:")
    display(
        accounts["status"]
        .value_counts(dropna=False)
        .to_frame("Accounts")
    )


# 2. PAYMENT PERFORMANCE
payments = data["payments"].copy()

print("\n2. PAYMENT PERFORMANCE")
print("-" * 50)

print("Total payment rows:", len(payments))
print("Unique accounts with payments:", payments["account_id"].nunique())
print("Unique borrowers with payments:", payments["borrower_id"].nunique())

payments["amount"] = pd.to_numeric(
    payments["amount"], errors="coerce"
)

print(
    "Total payment amount:",
    round(payments["amount"].sum(), 2)
)

if "payment_status" in payments.columns:
    print("\nPayment status:")
    display(
        payments["payment_status"]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

    payment_status_summary = (
        payments.groupby("payment_status")["amount"]
        .agg(["count", "sum", "mean"])
        .reset_index()
        .sort_values("sum", ascending=False)
    )

    display(payment_status_summary)


# 3. CALL PERFORMANCE
calls = data["calls"].copy()

print("\n3. CALL PERFORMANCE")
print("-" * 50)

print("Total calls:", len(calls))
print("Unique accounts called:", calls["account_id"].nunique())
print("Unique borrowers called:", calls["borrower_id"].nunique())
print("Unique agents:", calls["agent_id"].nunique())

if "call_status" in calls.columns:
    print("\nCall status distribution:")
    display(
        calls["call_status"]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

if "direction" in calls.columns:
    print("\nCall direction:")
    display(
        calls["direction"]
        .value_counts(dropna=False)
        .to_frame("Count")
    )


# 4. CALL ATTEMPT PERFORMANCE
attempts = data["call_attempts"].copy()

print("\n4. CALL ATTEMPT PERFORMANCE")
print("-" * 50)

print("Total call attempts:", len(attempts))
print("Unique accounts attempted:", attempts["account_id"].nunique())
print("Unique borrowers attempted:", attempts["borrower_id"].nunique())

if "attempt_no" in attempts.columns:
    print("\nAttempt number distribution:")
    display(
        attempts["attempt_no"]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Attempts")
    )

if "attempt_status" in attempts.columns:
    print("\nAttempt status distribution:")
    display(
        attempts["attempt_status"]
        .value_counts(dropna=False)
        .to_frame("Count")
    )


# 5. CAMPAIGN OVERVIEW
campaigns = data["campaigns"].copy()

print("\n5. CAMPAIGN OVERVIEW")
print("-" * 50)

print("Total campaigns:", campaigns["campaign_id"].nunique())

if "channel" in campaigns.columns:
    print("\nCampaign channels:")
    display(
        campaigns["channel"]
        .value_counts(dropna=False)
        .to_frame("Campaigns")
    )

if "strategy_version" in campaigns.columns:
    print("\nStrategy versions:")
    display(
        campaigns["strategy_version"]
        .value_counts(dropna=False)
        .to_frame("Campaigns")
    )


# 6. DAILY TARGETING
targeting = data["daily_targeting"].copy()

print("\n6. DAILY TARGETING")
print("-" * 50)

print("Total targeting records:", len(targeting))
print("Unique targeted accounts:", targeting["account_id"].nunique())
print("Unique campaigns used:", targeting["campaign_id"].nunique())

if "recommended_channel" in targeting.columns:
    print("\nRecommended channels:")
    display(
        targeting["recommended_channel"]
        .value_counts(dropna=False)
        .to_frame("Targets")
    )

if "priority" in targeting.columns:
    print("\nTarget priority:")
    display(
        targeting["priority"]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("Targets")
    )

if "status" in targeting.columns:
    print("\nTargeting status:")
    display(
        targeting["status"]
        .value_counts(dropna=False)
        .to_frame("Targets")
    )


print("\n" + "=" * 70)
print("STEP 20 COMPLETE")
print("=" * 70)

BUSINESS PERFORMANCE BASELINE

1. ACCOUNT PORTFOLIO
--------------------------------------------------
Total accounts: 30000
Total borrowers: 10943
Total principal amount: 12103665653.54
Total outstanding amount: 10489035343.0

DPD distribution:


,Accounts
dpd,
0,2685
1,2713
5,2727
15,2736
30,2704
45,2744
60,2770
75,2741
90,2727



Risk segment distribution:


,Accounts
risk_segment,
HIGH,7552
MEDIUM,7533
LOW,7513
NPA,7402



Account status distribution:


...


Target priority:


,Targets
priority,
1,4531
2,4539
3,4520
4,4358
5,4401
6,4432
7,4641
8,4587
9,4539



Targeting status:


,Targets
status,
EXPIRED,11371
CONTACTED,11254
QUEUED,11202
SKIPPED,11173



STEP 20 COMPLETE


In [ ]:
print("=" * 75)
print("              FINAL DATA QUALITY & ANALYSIS SUMMARY")
print("=" * 75)

print("\n1. DATASET OVERVIEW")
print("-" * 75)

total_rows = sum(len(df) for df in golden.values())

print("Total datasets:", len(golden))
print("Total records across datasets:", f"{total_rows:,}")

print("\nDataset sizes:")
for name, df in golden.items():
    print(f"{name:25s} : {len(df):,} rows | {len(df.columns)} columns")


print("\n2. MISSING VALUE FINDINGS")
print("-" * 75)

print("Highest missing-value issues:")

for _, row in missing_df.head(10).iterrows():
    print(
        f"{row['Dataset']:20s} | "
        f"{row['Column']:20s} | "
        f"{row['Missing Values']:,} missing | "
        f"{row['Missing %']}%"
    )


print("\n3. DUPLICATE RECORD FINDINGS")
print("-" * 75)

for _, row in duplicate_df.iterrows():
    print(
        f"{row['Dataset']:20s} | "
        f"{row['Duplicate Rows']:,} duplicate rows | "
        f"{row['Duplicate %']}%"
    )


print("\n4. DUPLICATE KEY ANALYSIS")
print("-" * 75)

display(duplicate_key_df)


print("\n5. IDENTITY CONFLICT ANALYSIS")
print("-" * 75)

print("Agent ID conflicts:", 1000)
print("Agent identity conflict rate: 100.00%")

print("Borrower IDs with identity conflicts:", 8518)
print("Borrower identity conflict rate: 77.33%")


print("\n6. PAYMENT REFERENCE ANALYSIS")
print("-" * 75)

print("Total unique payment references:", 20821)
print("Conflicting payment references:", 3407)
print("Payment reference conflict rate: 16.36%")


print("\n7. TIMESTAMP QUALITY")
print("-" * 75)

print("Agent sessions with logout before login: 0")
print("Timestamp ordering issue detected: No")


print("\n8. CAMPAIGN TARGETING ANALYSIS")
print("-" * 75)

targeting = golden["daily_targeting"]

print("Total targeting records:", f"{len(targeting):,}")
print("Unique targeted accounts:", f"{targeting['account_id'].nunique():,}")
print("Unique campaigns used:", f"{targeting['campaign_id'].nunique():,}")

print("\nRecommended channels:")
print(targeting["recommended_channel"].value_counts())


print("\n9. KEY BUSINESS INSIGHTS")
print("-" * 75)

print("• Borrower identity consistency is a major data-quality concern.")
print("• Agent identity data shows complete identity conflict coverage.")
print("• Payment references contain significant cross-record conflicts.")
print("• Exact duplicate payment records represent 1.91% of payment rows.")
print("• Calls, borrowers, payments and WhatsApp events contain duplicate records.")
print("• Timestamp validation found no logout-before-login violations.")
print("• Campaign targeting contains 45,000 records across 120 campaigns.")


print("\n10. RECOMMENDATIONS")
print("-" * 75)

print("1. Establish stronger borrower identity validation rules.")
print("2. Enforce uniqueness and consistency checks on payment references.")
print("3. Investigate duplicate payment and transaction records before reporting.")
print("4. Standardize missing contact information in borrower records.")
print("5. Implement timestamp validation during data ingestion.")
print("6. Add automated duplicate and identity-conflict monitoring.")
print("7. Use campaign/channel analysis to improve targeting decisions.")


print("\n" + "=" * 75)
print("             DATA QUALITY ASSESSMENT COMPLETE")
print("=" * 75)

              FINAL DATA QUALITY & ANALYSIS SUMMARY

1. DATASET OVERVIEW
---------------------------------------------------------------------------
Total datasets: 18
Total records across datasets: 639,328

Dataset sizes:
call_dispositions         : 35,000 rows | 8 columns
sms_events                : 45,000 rows | 8 columns
payments                  : 25,500 rows | 9 columns
accounts                  : 30,000 rows | 11 columns
agent_sessions            : 15,000 rows | 7 columns
account_status_history    : 60,000 rows | 8 columns
field_visits              : 25,000 rows | 10 columns
call_attempts             : 120,000 rows | 9 columns
calls                     : 91,350 rows | 11 columns
agents                    : 30,000 rows | 8 columns
complaints                : 8,000 rows | 9 columns
borrowers                 : 30,600 rows | 8 columns
campaigns                 : 120 rows | 7 columns
vendor_telephony          : 15 rows | 6 columns
daily_targeting           : 45,000 rows | 7 columns
w

,Dataset,Column,Duplicate IDs,Unique IDs,Total Rows,Duplicate %
0,borrowers,borrower_id,19585,11015,30600,64.00
1,payments,payment_id,500,25000,25500,1.96
2,payments,payment_reference,4678,20821,25500,18.35
3,calls,call_id,1350,90000,91350,1.48
4,whatsapp_events,whatsapp_event_id,600,60000,60600,0.99



5. IDENTITY CONFLICT ANALYSIS
---------------------------------------------------------------------------
Agent ID conflicts: 1000
Agent identity conflict rate: 100.00%
Borrower IDs with identity conflicts: 8518
Borrower identity conflict rate: 77.33%

6. PAYMENT REFERENCE ANALYSIS
---------------------------------------------------------------------------
Total unique payment references: 20821
Conflicting payment references: 3407
Payment reference conflict rate: 16.36%

7. TIMESTAMP QUALITY
---------------------------------------------------------------------------
Agent sessions with logout before login: 0
Timestamp ordering issue detected: No

8. CAMPAIGN TARGETING ANALYSIS
---------------------------------------------------------------------------
Total targeting records: 45,000
Unique targeted accounts: 23,344
Unique campaigns used: 120

Recommended channels:
recommended_channel
FIELD       11365
SMS         11221
WHATSAPP    11212
VOICE       11202
Name: count, dtype: int64

9. 